In [ ]:
# If you haven’t done so already, install the fastf1 library
!pip install fastf1


In [ ]:
# Import the libraries required for code execution
import numpy as np
import pandas as pd
import fastf1
import datetime as dt


## Events

In [ ]:
# Create the lists that will contain the event data
EventKey = []
Country = []
EventName = []

# For loop to extract data from all 2025 events
for i in range(1, 25):
    event = fastf1.get_event(2025, i)
    EventKey.append(event.RoundNumber)
    Country.append(event.Country)
    EventName.append(event.EventName)

# Create a DataFrame from the lists and save it as a CSV file
Events = pd.DataFrame({
    "EventKey": EventKey,
    "Country": Country,
    "EventName": EventName
})

Events.to_csv(r'Events.csv', index=False)



req         WARNING 	DEFAULT CACHE ENABLED! (3.02 GB) C:\Users\aless\AppData\Local\Temp\fastf1


## Tyres

In [ ]:
# Manually create the dictionary of F1 tyres and save it as a CSV file

Tyres = {1: 'C1', 2: 'C2', 3: 'C3', 4: 'C4', 5: 'C5', 6: 'C6', 7: 'INT', 8: 'WET'}

Tyres_df = pd.DataFrame.from_dict(Tyres, orient='index', columns=['Tyre'])

Tyres_df.to_csv('Tyres.csv', index_label='TyreKey')


## Drivers

In [ ]:
# Create the list that will contain the DataFrames for each event
Events = []

# For loop to extract data for all drivers who participated in at least one race in 2025
for i in range(1, 20):
    Event = fastf1.get_session(2025, i, 'R')
    Event.load()
    Drivers_Event = Event.results

    # Reorganize the columns of the DataFrame and add it to the list
    Drivers_Event = Drivers_Event.loc[:, ['DriverNumber', 'Abbreviation', 'TeamName', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl']]
    Events.append(Drivers_Event)

# Concatenate all DataFrames into one, remove duplicates, and create the DriverKey column
Drivers = pd.concat(Events, ignore_index=True)
Drivers = Drivers.drop_duplicates(subset=['DriverNumber', 'TeamName'])
Drivers['DriverKey'] = range(1, len(Drivers) + 1)

# Reorganize the columns and save to CSV
Drivers = Drivers.loc[:, ['DriverKey', 'DriverNumber', 'Abbreviation', 'TeamName', 
                          'FirstName', 'LastName', 'FullName', 'HeadshotUrl']]

Drivers.to_csv(r'Drivers_2025.csv', index=False)


core           INFO 	Loading data for Australian Grand Prix - Race [v3.5.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 4 completed the race distance 00:00.022000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 drivers: ['4', '1', '63', '12', '23', '18', '27', '16', '81', '44', '10', 

## Team

In [ ]:
# Extract the names of the 2025 teams, add the TeamKey column, and save to CSV
team = fastf1.get_session(2025, 'Australia', 'Q')
team.load()
team = team.results
team = team.loc[:, ['TeamName']].drop_duplicates()
team = team.reset_index(drop=True)
team['TeamKey'] = range(1, len(team) + 1)
team = team[['TeamKey', 'TeamName']]
team.to_csv('Team_2025.csv', index=False)


## Race Laps

In [ ]:
# Create the list that will contain the DataFrames for each race
race_list = []

# For loop to extract data for every race held in 2025 (max 25 in the range, but there are 24 races)
for i in range(1, 20):
    race = fastf1.get_session(2025, i, 'R')
    race.load()
    race = race.laps

    # Add the EventKey column and append the DataFrame to the list
    race['EventKey'] = i
    race_list.append(race)

# Create the dictionary of tyres used in each race and convert it to a DataFrame
Tyre = {
    'EventKey': [i for i in range(1, 20) for _ in range(5)], 
    'Compound': ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET'] * 18, 
    'TyreKey': [5, 4, 3, 7, 8, 4, 3, 2, 7, 8, 3, 2, 1, 7, 8, 3, 2, 1, 7, 8, 5, 4, 3, 7, 8, 5, 4, 3, 7, 8, 6, 5, 4, 7, 8, 6, 5, 4, 7, 8,
                3, 2, 1, 7, 8, 6, 5, 4, 7, 8, 5, 4, 3, 7, 8, 4, 3, 2, 7, 8, 4, 3, 1, 7, 8, 5, 4, 3, 7, 8, 4, 3, 2, 7, 8, 5, 4, 3, 7, 8,
                6, 5, 4, 7, 8, 5, 4, 3, 7, 8, 1, 3, 4, 7, 8]
}

Tyre_df = pd.DataFrame(Tyre)    

# Concatenate all race DataFrames into one
race_laps = pd.concat(race_list, ignore_index=True)

# Add the TyreKey column via merge
race_laps = pd.merge(race_laps, Tyre_df, on=['EventKey', 'Compound'], how='left')

# Load the drivers DataFrame and add the DriverKey column via merge
drivers = pd.read_csv(r'Drivers_2025.csv')

# Adjust data types for merging
race_laps['DriverNumber'] = race_laps['DriverNumber'].astype('int64')
race_laps = pd.merge(race_laps, drivers[['DriverNumber', 'DriverKey']], on='DriverNumber', how='left')

# Handle cases where drivers changed teams during the season
mask = race_laps["DriverNumber"] == 30
race_laps.loc[mask, "DriverKey"] = np.where(
    race_laps.loc[mask, "EventKey"] > 2,
    22,
    15
)

mask = race_laps["DriverNumber"] == 22
race_laps.loc[mask, "DriverKey"] = np.where(
    race_laps.loc[mask, "EventKey"] > 2,
    21,
    12
)

# Reorganize columns and adjust the data types of time columns
race_laps = race_laps.loc[:, ['DriverKey', 'EventKey', 'LapNumber', 'Position', 'LapTime',
                              'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time',
                              'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'TyreKey', 'Compound',
                              'Stint', 'TyreLife', 'TrackStatus']]

# Convert time columns to total seconds
race_laps[['LapTime', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']] = race_laps[['LapTime', 'PitOutTime',
                                                                                                             'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']].apply(pd.to_timedelta)

race_laps[['LapTime', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time']] = race_laps[['LapTime', 'PitOutTime', 'PitInTime', 'Sector1Time', 
                                                                                                            'Sector2Time', 'Sector3Time']].apply(lambda x: pd.to_timedelta(x).dt.total_seconds())

# Save to CSV
race_laps.to_csv(r'Race_laps.csv', index=False)


## Results

In [ ]:
# Create the list that will contain the DataFrames for each event
event_list = []

# For loop to extract data for each event held in 2025 (max 25 in the range, but there are 24 races)
for i in range(1, 20):

    # Extract qualifying data
    Quali = fastf1.get_session(2025, i, 'Q')
    Quali.load()
    results_Q = Quali.results 
    # Select the relevant columns
    results_Q = results_Q.loc[:, ['DriverNumber', 'Position', 'Q1', 'Q2', 'Q3']]

    # Extract race data
    Race = fastf1.get_session(2025, i, 'R')
    Race.load()
    results_R = Race.results
    # Select the relevant columns
    results_R = results_R.loc[:, ['DriverNumber', 'Position', 'Time', 'Status', 'Points']]

    # Merge the two DataFrames
    results_event = pd.merge(results_Q, results_R, on='DriverNumber', how='outer')

    # Add the EventKey column, rename columns, and convert time columns
    results_event['EventKey'] = i
    results_event.columns = ['DriverNumber', 'GridPosition', 'Q1', 'Q2', 'Q3', 'RacePosition', 'RaceTime', 'Status', 'Points', 'EventKey']
    results_event['Q1'] = results_event['Q1'].dt.total_seconds()
    results_event['Q2'] = results_event['Q2'].dt.total_seconds()
    results_event['Q3'] = results_event['Q3'].dt.total_seconds()
    results_event['RaceTime'] = results_event['RaceTime'].dt.total_seconds()

    # Sort the data by the final race position
    results_event = results_event.sort_values(by='RacePosition')

    # Create a column with each driver's best qualifying time by combining Q3, Q2, and Q1
    results_event['QualiTime'] = results_event[['Q3', 'Q2', 'Q1']].bfill(axis=1).iloc[:, 0]

    # Create a column with the difference between each driver’s qualifying time and the pole position time
    best_Q3 = results_event['Q3'].min(skipna=True)
    results_event['QualiDiff'] = results_event['QualiTime'] - best_Q3

    # Add the event DataFrame to the list
    event_list.append(results_event)

# Concatenate all event DataFrames into a single one
results = pd.concat(event_list, ignore_index=True)

# Adjust the data type of the DriverNumber column to enable merging
results['DriverNumber'] = results['DriverNumber'].astype('int64')

# Load the drivers DataFrame and add the DriverKey column via merge
drivers = pd.read_csv(r'Drivers_2025.csv')
res = pd.merge(results, drivers[['DriverNumber', 'DriverKey']], on='DriverNumber', how='left')

# Handle cases where drivers changed teams during the season
mask1 = res["DriverNumber"] == 30
res.loc[mask1, "DriverKey"] = np.where(
    res.loc[mask1, "EventKey"] > 2,
    22,
    15
)

mask2 = res["DriverNumber"] == 22
res.loc[mask2, "DriverKey"] = np.where(
    res.loc[mask2, "EventKey"] > 2,
    21,
    12
)

# Remove duplicates, reorder columns, and select the relevant ones
res = res.drop_duplicates()
res = res.loc[:, ['DriverKey', 'EventKey', 'RacePosition', 'Points', 'RaceTime', 'Status',
                  'GridPosition', 'Q1', 'Q2', 'Q3', 'QualiTime', 'QualiDiff']]

# Save to CSV
res.to_csv(r'Results_2025.csv')


## Telemetry

In [ ]:
# Create the list that will contain the DataFrames for each driver in each event
telemetry = []

# For loop to extract data for each event held in 2025 (max 25 in the range, but there are 24 races)
for i in range(1, 20):

    # Extract all laps data from qualifying
    qualifying = fastf1.get_session(2025, i, 'Q')
    qualifying.load()
    qualifying = qualifying.laps

    # Loop through each driver to extract telemetry for their fastest lap, skipping drivers without telemetry
    for driver in qualifying['Driver'].unique():
        try:
            driver_laps = qualifying.pick_drivers(driver)
            driver_telemetry = driver_laps.pick_fastest().get_telemetry()
            driver_telemetry['Driver'] = driver

            # Add EventKey column, convert Time column to total seconds, and select relevant columns
            driver_telemetry['EventKey'] = i  
            driver_telemetry['Distance'] = driver_telemetry['Distance'].round(0)
            driver_telemetry['Time'] = driver_telemetry['Time'].dt.total_seconds()
            driver_telemetry = driver_telemetry.loc[:, ['Driver', 'EventKey', 'Time', 'RPM', 'Speed', 'nGear', 'Throttle', 
                                                        'Brake', 'DRS', 'Distance', 'RelativeDistance', 'X', 'Y', 'Z']]

            # Append the driver DataFrame to the list
            telemetry.append(driver_telemetry)
        except:
            print(f"Driver {driver} has no telemetry data.")

# Concatenate all drivers' DataFrames into a single DataFrame
telemetry = pd.concat(telemetry, ignore_index=True)

# Load the drivers DataFrame and add the DriverKey column via merge
drivers = pd.read_csv(r'Piloti_2025.csv')
tel = pd.merge(telemetry, drivers[['Abbreviation', 'DriverKey']], left_on='Driver', right_on='Abbreviation', how='left')

# Handle cases where drivers changed teams during the season
mask1 = tel["Driver"] == 'LAW'
tel.loc[mask1, "DriverKey"] = np.where(
    tel.loc[mask1, "EventKey"] > 2,
    22,
    15
)

mask2 = tel["Driver"] == 'TSU'
tel.loc[mask2, "DriverKey"] = np.where(
    tel.loc[mask2, "EventKey"] > 2,
    21,
    12
)

# Remove duplicates, reorder columns, select relevant ones, and save to CSV
tel = tel.drop_duplicates()
tel = tel.loc[:, ['DriverKey', 'EventKey', 'Time',
                  'RPM', 'Speed', 'nGear', 'Throttle', 'Brake', 'DRS', 
                  'Distance', 'RelativeDistance']]
tel.to_csv(r'Telemetry_Q_2025.csv', index=False)

